# Parakeet SRT Transcriber — Google Colab

ColabのGPUランタイムを選び、このノートブックを実行してください。セルの開始時に公開GitHubリポジトリからアプリ本体の最新コードを取得します。ZIPのアップロードは不要です。メディアファイルだけColabのファイルマネージャーから `/content` にアップロードしてください。

この1セルでアプリ本体の取得、依存関係のセットアップ、Webサーバー起動、画面表示まで行います。MOSS生成上限は5120トークンです。初回は依存関係とモデルのダウンロードに時間がかかります。同じランタイム内でセルを停止・再実行しても、依存関係は再インストールせず、処理途中のデータも保持します。ファイル、モデル、処理途中のデータはColabの一時ストレージに保存され、ランタイム終了後は消えます。
実行セルはサーバーを監視するため実行中のままになります。アプリを終了するときはColabの停止ボタンを押してください。Colab側の最大実行時間やランタイム終了までは防げません。


In [ ]:
from pathlib import Path
import json
import hashlib
import os
import shutil
import subprocess
import sys
import time
import urllib.request

# 公開リポジトリからアプリ本体を取得・更新します。
repo_url = 'https://github.com/Elaina117/Parakeet_SRT_Transcriber.git'
repo_branch = 'main'
app_dir = Path('/content/Parakeet_SRT_Transcriber')
checkout_dir = Path('/content/Parakeet_SRT_Transcriber_source')

# 実行し直した場合は前のサーバーを終了してからソースを更新します。
old_server = globals().get('server_process')
if old_server is not None and old_server.poll() is None:
    old_server.terminate()
    try:
        old_server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        old_server.kill()

subprocess.run(['nvidia-smi'], check=True)
if (app_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(app_dir), 'pull', '--ff-only', 'origin', repo_branch], check=True)
else:
    # Migrate ZIP-era /content data without deleting checkpoints, models, or installed runtimes.
    if checkout_dir.exists():
        shutil.rmtree(checkout_dir)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', repo_branch, repo_url, str(checkout_dir)], check=True)
    if app_dir.exists() and any(app_dir.iterdir()):
        for source in checkout_dir.iterdir():
            if source.name == '.git':
                continue
            destination = app_dir / source.name
            if source.is_dir():
                shutil.copytree(source, destination, dirs_exist_ok=True)
            else:
                shutil.copy2(source, destination)
        shutil.move(str(checkout_dir / '.git'), str(app_dir / '.git'))
        shutil.rmtree(checkout_dir)
    else:
        if app_dir.exists():
            app_dir.rmdir()
        shutil.move(str(checkout_dir), str(app_dir))
if not (app_dir / 'server.py').is_file():
    raise RuntimeError('GitHubからアプリ本体を取得できませんでした。公開リポジトリの設定を確認してください。')

# Colab/Linux用CUDA依存関係を必要なときだけ導入します。
def install(*args):
    print('pip:', ' '.join(args), flush=True)
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

minor = sys.version_info.minor
sherpa_versions = {
    10: '1.13.8+cuda12.cudnn9.onnxruntime1.28.2',
    11: '1.13.8+cuda12.cudnn9.onnxruntime1.28.2',
    12: '1.13.8+cuda12.cudnn9.onnxruntime1.28.2',
    13: '1.13.1+cuda12.cudnn9',
    14: '1.13.7+cuda12.cudnn9.onnxruntime1.27.1',
}
if minor not in sherpa_versions:
    raise RuntimeError(f'Python 3.{minor}用のCUDA Sherpa-ONNX wheelが未設定です')
sherpa_spec = 'sherpa-onnx==' + sherpa_versions[minor]
moss_env = app_dir / '.moss-venv'
setup_marker = app_dir / '.colab-setup-fingerprint'
requirements_hash = hashlib.sha256((app_dir / 'requirements-colab.txt').read_bytes()).hexdigest()
setup_fingerprint = f'colab-setup-v2|py{minor}|{sherpa_spec}|{requirements_hash}'
setup_ready = (
    setup_marker.is_file()
    and setup_marker.read_text(encoding='utf-8').strip() == setup_fingerprint
    and (moss_env / 'bin' / 'python').is_file()
    and (moss_env / '.install-complete').is_file()
)
# Adopt a working environment made by the previous notebook version, which
# did not write a fingerprint but did finish installing before starting server.
if not setup_ready and not setup_marker.exists() and (moss_env / '.install-complete').is_file():
    try:
        from importlib.metadata import version
        essential_distributions = (
            'fastapi', 'uvicorn', 'python-multipart', 'numpy', 'imageio-ffmpeg',
            'onnx-asr', 'huggingface-hub', 'onnx',
        )
        for distribution in essential_distributions:
            version(distribution)
        setup_ready = (
            version('onnxruntime-gpu') == '1.26.0'
            and version('sherpa-onnx').startswith(sherpa_versions[minor].split('+')[0])
            and (moss_env / 'bin' / 'python').is_file()
        )
        if setup_ready:
            setup_marker.write_text(setup_fingerprint + '\n', encoding='utf-8')
    except Exception:
        setup_ready = False
if setup_ready:
    print('依存関係はセットアップ済みのため、インストールをスキップします。', flush=True)
else:
    install('install', '-r', str(app_dir / 'requirements-colab.txt'))
    install('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
    install('install', 'onnxruntime-gpu==1.26.0')
    install('install', '--no-deps', sherpa_spec, '-f', 'https://k2-fsa.github.io/sherpa/onnx/cuda.html')

    # MOSSはColabのPyTorch/CUDA環境を共有する分離venvに導入します。
    if not (moss_env / 'bin' / 'python').is_file():
        try:
            subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(moss_env)], check=True)
        except subprocess.CalledProcessError:
            print('標準venvの作成に失敗したため、virtualenv方式へ切り替えます。', flush=True)
            if moss_env.exists():
                shutil.rmtree(moss_env)
            install('install', 'virtualenv')
            subprocess.run([sys.executable, '-m', 'virtualenv', '--system-site-packages', str(moss_env)], check=True)
    moss_pip = moss_env / 'bin' / 'pip'
    subprocess.run([str(moss_pip), 'install', '--upgrade', 'pip'], check=True)
    subprocess.run([str(moss_pip), 'install', 'git+https://github.com/OpenMOSS/MOSS-Transcribe-Diarize.git'], check=True)
    (moss_env / '.install-complete').write_text('installed\n', encoding='utf-8')
    setup_marker.write_text(setup_fingerprint + '\n', encoding='utf-8')

# CUDAが利用できることを確かめてからアプリを起動します。
import torch  # Load the Colab CUDA/cuDNN runtime before ONNX Runtime.
import onnxruntime as ort
print('PyTorch:', torch.__version__, 'CUDA runtime:', torch.version.cuda, 'available:', torch.cuda.is_available())
print('ONNX Runtime:', ort.__version__, 'providers:', ort.get_available_providers())
if not torch.cuda.is_available() or 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('GPU/CUDAが見つかりません。ColabでGPUランタイムを選択してからセルを実行してください。')

env = os.environ.copy()
env['PARAKEET_COLAB'] = '1'
env['MOSS_MAX_NEW_TOKENS'] = '5120'  # Windows版と同じMOSS生成上限
log_path = Path('/content/parakeet_server.log')
server_log = log_path.open('w')
server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '5173'],
    cwd=app_dir, env=env, stdout=server_log, stderr=subprocess.STDOUT,
)
# モデルのロード完了を待たずに画面を開き、進捗を表示できるようにします。
for _ in range(120):
    if server_process.poll() is not None:
        print(log_path.read_text(errors='replace')[-8000:])
        raise RuntimeError('Webサーバーが終了しました。上のログを確認してください。')
    try:
        with urllib.request.urlopen('http://127.0.0.1:5173/api/status', timeout=2) as response:
            json.load(response)
        break
    except Exception:
        time.sleep(1)
else:
    print(log_path.read_text(errors='replace')[-8000:])
    raise RuntimeError('Webサーバーの起動がタイムアウトしました。')

from google.colab import output
output.serve_kernel_port_as_iframe(5173, height='1200')
print('起動しました。動画は /content/動画ファイル名 の形式で指定してください。')

# セルが終了するとColabでアイドル扱いになるため、サーバー稼働中はこのセルで監視を続けます。
try:
    print('サーバー稼働中。このセルはアプリを使っている間そのままにしてください。', flush=True)
    next_heartbeat = time.monotonic() + 300
    while server_process.poll() is None:
        time.sleep(10)
        if time.monotonic() >= next_heartbeat:
            print(f'サーバー稼働中 ({time.strftime("%H:%M:%S")})', flush=True)
            next_heartbeat = time.monotonic() + 300
    server_log.flush()
    raise RuntimeError('Webサーバーが終了しました。ログ:\n' + log_path.read_text(errors='replace')[-8000:])
except KeyboardInterrupt:
    print('停止要求を受け取りました。Webサーバーを終了します。', flush=True)
finally:
    if server_process.poll() is None:
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            server_process.kill()
            server_process.wait()
    server_log.close()
